# 🌊 FloodNet CNN Training v2.0

Complete training pipeline for Odisha Flood Validation System.

**Features:**
- Downloads real flood images from reliable CDNs
- Generates synthetic augmented samples
- Trains MobileNetV2 for binary flood classification
- Exports model for local deployment

**Instructions:**
1. Set Runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Download the trained `flood_cnn_v2.pth` model

In [ ]:
# Cell 1: Environment Setup
import os
import shutil
import random
import ssl
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
from PIL import Image, ImageDraw, ImageFilter

# Disable SSL verification for some CDNs
ssl._create_default_https_context = ssl._create_unverified_context

print(f"✅ PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Configuration
DATA_DIR = Path("data/flood_images")
FLOOD_DIR = DATA_DIR / "flood"
NOT_FLOOD_DIR = DATA_DIR / "not_flood"
MODEL_SAVE_PATH = "flood_cnn_v2.pth"

# Training hyperparameters
TARGET_SAMPLES = 150  # Per class
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

print(f"📁 Data directory: {DATA_DIR.absolute()}")
print(f"🎯 Target samples per class: {TARGET_SAMPLES}")
print(f"⚙️  Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}")

In [ ]:
# Cell 3: Dataset Generation Functions

def setup_directories():
    """Clean and create data directories."""
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    FLOOD_DIR.mkdir(parents=True, exist_ok=True)
    NOT_FLOOD_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Created directories")

def download_file(url: str, save_path: Path) -> bool:
    """Download with fallback."""
    headers = {'User-Agent': 'Mozilla/5.0 FloodNet-Colab/2.0'}
    try:
        request = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(request, timeout=15) as response:
            with open(save_path, 'wb') as f:
                f.write(response.read())
        return True
    except Exception as e:
        return False

def generate_flood_image(index: int) -> Path:
    """Generate synthetic flood image (muddy water scene)."""
    save_path = FLOOD_DIR / f"synth_flood_{index:03d}.jpg"
    img = Image.new('RGB', (224, 224))
    draw = ImageDraw.Draw(img)
    
    # Gray sky
    for y in range(0, 80):
        gray = 150 + random.randint(-20, 20)
        draw.line([(0, y), (224, y)], fill=(gray, gray, gray + 30))
    
    # Brown/muddy flood water
    for y in range(80, 224):
        r = 100 + random.randint(-30, 30)
        g = 80 + random.randint(-20, 20)
        b = 60 + random.randint(-20, 20)
        draw.line([(0, y), (224, y)], fill=(r, g, b))
    
    # Debris
    for _ in range(20):
        x, y = random.randint(0, 200), random.randint(90, 220)
        w, h = random.randint(5, 25), random.randint(3, 8)
        draw.ellipse([x, y, x+w, y+h], fill=(60, 50, 40))
    
    img = img.filter(ImageFilter.GaussianBlur(radius=1.5))
    img.save(save_path, quality=90)
    return save_path

def generate_normal_image(index: int) -> Path:
    """Generate synthetic normal scene (clear day)."""
    save_path = NOT_FLOOD_DIR / f"synth_normal_{index:03d}.jpg"
    img = Image.new('RGB', (224, 224))
    draw = ImageDraw.Draw(img)
    
    # Blue sky
    for y in range(0, 100):
        b = 200 + random.randint(-20, 20)
        draw.line([(0, y), (224, y)], fill=(135, 180, b))
    
    # Green ground
    for y in range(100, 224):
        g = 120 + random.randint(-30, 30)
        draw.line([(0, y), (224, y)], fill=(60, g, 50))
    
    # Trees/buildings
    for _ in range(5):
        x = random.randint(10, 200)
        y = random.randint(100, 180)
        w, h = random.randint(10, 30), random.randint(20, 60)
        color = (40, 80 + random.randint(-20, 20), 40)
        draw.rectangle([x, y - h, x + w, y], fill=color)
    
    img = img.filter(ImageFilter.GaussianBlur(radius=1))
    img.save(save_path, quality=90)
    return save_path

print("✅ Dataset generation functions loaded")

In [ ]:
# Cell 4: Generate Dataset
print("="*50)
print("🌊 GENERATING FLOODNET DATASET")
print("="*50)

setup_directories()

# Try to download some real images
real_urls = {
    "flood": [
        ("https://images.unsplash.com/photo-1547683905-f686c993aae5?w=400", "real_flood_0.jpg"),
        ("https://images.unsplash.com/photo-1562155618-e1a8bc2eb04f?w=400", "real_flood_1.jpg"),
    ],
    "normal": [
        ("https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400", "real_normal_0.jpg"),
        ("https://images.unsplash.com/photo-1501785888041-af3ef285b470?w=400", "real_normal_1.jpg"),
    ]
}

print("\n📥 Downloading real images...")
for url, name in real_urls["flood"]:
    if download_file(url, FLOOD_DIR / name):
        print(f"  ✓ {name}")
for url, name in real_urls["normal"]:
    if download_file(url, NOT_FLOOD_DIR / name):
        print(f"  ✓ {name}")

# Generate synthetic samples
print(f"\n🎨 Generating {TARGET_SAMPLES} flood images...")
for i in range(TARGET_SAMPLES):
    generate_flood_image(i)
    if (i + 1) % 50 == 0:
        print(f"  Progress: {i + 1}/{TARGET_SAMPLES}")

print(f"\n🏠 Generating {TARGET_SAMPLES} normal images...")
for i in range(TARGET_SAMPLES):
    generate_normal_image(i)
    if (i + 1) % 50 == 0:
        print(f"  Progress: {i + 1}/{TARGET_SAMPLES}")

# Summary
flood_count = len(list(FLOOD_DIR.glob("*.jpg")))
normal_count = len(list(NOT_FLOOD_DIR.glob("*.jpg")))
print(f"\n📊 Dataset Summary:")
print(f"  🌊 Flood images: {flood_count}")
print(f"  🏠 Normal images: {normal_count}")
print(f"  📦 Total: {flood_count + normal_count}")

In [ ]:
# Cell 5: Dataset Class

class FloodDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        # Load flood images (label=1)
        flood_dir = self.root_dir / "flood"
        for img_path in flood_dir.glob("*.jpg"):
            self.samples.append((str(img_path), 1.0))
        
        # Load non-flood images (label=0)
        normal_dir = self.root_dir / "not_flood"
        for img_path in normal_dir.glob("*.jpg"):
            self.samples.append((str(img_path), 0.0))
        
        print(f"📦 Loaded {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception:
            img = torch.zeros((3, 224, 224))
        return img, torch.tensor(label, dtype=torch.float32)

print("✅ FloodDataset class loaded")

In [ ]:
# Cell 6: Model Definition

def create_model():
    """Create MobileNetV2 with binary classification head."""
    model = models.mobilenet_v2(weights='IMAGENET1K_V1')
    
    # Freeze feature extractor
    for param in model.features.parameters():
        param.requires_grad = False
    
    # Replace classifier for binary output
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 1),
        nn.Sigmoid()
    )
    
    return model

print("✅ Model architecture defined")

In [ ]:
# Cell 7: Training Loop

def train_model():
    # Data transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Load dataset
    dataset = FloodDataset(DATA_DIR, transform=transform)
    
    if len(dataset) == 0:
        raise ValueError("No images found! Run Cell 4 first.")
    
    # Split data
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_data, val_data = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=2)
    
    print(f"📊 Train: {len(train_data)}, Val: {len(val_data)}")
    
    # Create model
    model = create_model().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    
    print("\n🚀 Starting training...")
    print("-" * 50)
    
    best_val_loss = float('inf')
    
    for epoch in range(EPOCHS):
        # Training phase
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze()
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
              f"Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Train Acc: {train_acc:.1f}% | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | "
              f"Val Acc: {val_acc:.1f}%")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
        
        scheduler.step()
    
    print("-" * 50)
    print(f"✅ Training complete! Best model saved to: {MODEL_SAVE_PATH}")
    return model

# Run training
model = train_model()

In [ ]:
# Cell 8: Download Trained Model
print("="*50)
print("📥 DOWNLOAD YOUR TRAINED MODEL")
print("="*50)

import os
if os.path.exists(MODEL_SAVE_PATH):
    file_size = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
    print(f"\n✅ Model file: {MODEL_SAVE_PATH}")
    print(f"   Size: {file_size:.2f} MB")
    print("\n📱 Click the download button below:")
    
    from google.colab import files
    files.download(MODEL_SAVE_PATH)
else:
    print("❌ Model file not found. Run training first!")

---
## 🎉 Done!

After downloading `flood_cnn_v2.pth`, place it in your project:
```
models/flood_cnn.pth
```

The Odisha Flood Validation System will automatically use your trained model!